# Mexico City GP 2026

Podium prediction for this race, made after qualifying. The notebook does four things:

1. downloads past results, weather and practice times
2. builds a few inputs for each driver (starting spot, recent form, and so on)
3. tests several scikit-learn models on past races and keeps the best one
4. uses that model on this weekend's grid

This one was re-run after the race with the final version of the code, but it only uses data from before the race.

Only the settings cell below changes from race to race.

In [1]:
%pip -q install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Settings

In [2]:
RACE_NAME = 'Mexico City GP 2026'
RACE_DATE = '2026-11-01'          # only races before this date are used
CIRCUIT = 'rodriguez'             # circuitId in the Ergast data
QUALI_KEY = 11400                 # OpenF1 session keys for this weekend
PRACTICE_KEYS = [11397, 11398, 11399]
PENALTIES = {'VER': 5, 'LAW': 9, 'ANT': 4, 'LIN': 8}   # driver code -> new starting slot
OUTPUT_NAME = 'mexico_2026'

In [3]:
import json
import time
import warnings
from collections import defaultdict, deque
from pathlib import Path

import fastf1
import numpy as np
import pandas as pd
import requests
from fastf1.ergast import Ergast
from sklearn.ensemble import (ExtraTreesClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
fastf1.set_log_level('ERROR')
CACHE = Path('cache')
CACHE.mkdir(exist_ok=True)

## Past results

Race results come from Jolpica (the Ergast API) through fastf1, 2022 up to the race date set above. 2022 is only used to build form for the 2023 races. One row is one driver in one race.

In [4]:
ergast = Ergast(limit=100)

def season_results(year):
    path = CACHE / f'results_{year}_{pd.Timestamp.now():%Y%m%d}.pkl'
    if path.exists():
        return pd.read_pickle(path)
    page = ergast.get_race_results(season=year)
    frames, got = [], 0
    while True:
        for (_, meta), result in zip(page.description.iterrows(), page.content):
            frame = pd.DataFrame(result)
            for col in ['season', 'round', 'raceDate', 'circuitId', 'raceName']:
                frame[col] = meta[col]
            frames.append(frame)
            got += len(frame)
        if got >= page.total_results:
            break
        time.sleep(0.4)
        try:
            page = page.get_next_result_page()
        except ValueError:
            break
    out = pd.concat(frames).drop_duplicates(['season', 'round', 'driverId'])
    out.to_pickle(path)
    return out

races = pd.concat([season_results(y) for y in range(2022, 2027)], ignore_index=True)
races['raceDate'] = pd.to_datetime(races['raceDate']).dt.tz_localize(None)
for col in ['position', 'points', 'grid']:
    races[col] = pd.to_numeric(races[col], errors='coerce')
races = races[races['raceDate'] < pd.Timestamp(RACE_DATE)]
races = races.sort_values(['raceDate', 'round', 'position'])
print(races.groupby('season')['round'].nunique().to_dict(), 'races per season')

{2022: 22, 2023: 22, 2024: 24, 2025: 24, 2026: 18} races per season


## Weather and practice

OpenF1 has the session data from 2023. For every past weekend I take two extra numbers:

- `quali_rain`: share of the qualifying session where it was raining (0 to 1)
- `practice_gap`: a driver's best practice lap, in seconds behind the fastest lap of that session

The results are cached on disk, so only the first run is slow.

In [5]:
def openf1(endpoint, **params):
    name = endpoint + '_' + '_'.join(f'{k}{v}' for k, v in sorted(params.items()))
    path = CACHE / (name.replace('/', '-') + '.json')
    if path.exists():
        return pd.DataFrame(json.loads(path.read_text()))
    for wait in (1, 3, 8, 20):
        data = requests.get('https://api.openf1.org/v1/' + endpoint, params=params).json()
        if isinstance(data, list):
            break
        time.sleep(wait)
    else:
        data = []
    path.write_text(json.dumps(data))
    time.sleep(0.35)
    return pd.DataFrame(data)


def practice_gaps(session_keys):
    best = {}
    for key in session_keys:
        result = openf1('session_result', session_key=key)
        if result.empty:
            continue
        lap = pd.to_numeric(result['duration'], errors='coerce')
        for number, gap in zip(result['driver_number'], lap - lap.min()):
            if pd.notna(gap):
                best[number] = min(best.get(number, 99), gap)
    return best


def rain_share(session_key):
    weather = openf1('weather', session_key=session_key)
    return float((weather['rainfall'] > 0).mean()) if len(weather) else np.nan


weekend = {}   # (season, round) -> (rain share, practice gaps)
for year in range(2023, 2027):
    sessions = openf1('sessions', year=year)
    for race in sessions[(sessions['session_name'] == 'Race') & (~sessions['is_cancelled'])].itertuples():
        day = pd.Timestamp(race.date_start[:10])
        match = races[(races['season'] == year) & ((races['raceDate'] - day).abs() <= pd.Timedelta(days=1))]
        if match.empty:
            continue
        meeting = sessions[sessions['meeting_key'] == race.meeting_key]
        quali = meeting[meeting['session_name'] == 'Qualifying']
        practice = meeting[meeting['session_name'].str.startswith('Practice')]
        rain = rain_share(int(quali['session_key'].iloc[0])) if len(quali) else np.nan
        weekend[(year, int(match['round'].iloc[0]))] = (rain, practice_gaps(practice['session_key']))

keys = list(zip(races['season'], races['round']))
races['quali_rain'] = [weekend.get(k, (np.nan, {}))[0] for k in keys]
races['practice_gap'] = [weekend.get(k, (np.nan, {}))[1].get(int(n), np.nan) if pd.notna(n) else np.nan
                         for k, n in zip(keys, pd.to_numeric(races['driverNumber'], errors='coerce'))]
print(races['quali_rain'].notna().mean().round(2), 'of rows have weather,',
      races['practice_gap'].notna().mean().round(2), 'have practice times')

0.8 of rows have weather, 0.78 have practice times


## Inputs for the model

For each driver in each race:

- `grid`: starting position
- recent form: average finish, average points and podium rate over the last 5 races, the team's points over its last 5 races, the driver's average finish here over the last 3 visits, and how many races of history we have
- `practice_gap` and `quali_rain` from above

Everything is built only from races before the one being predicted.

In [6]:
FORM = ['driver_finish_5', 'driver_points_5', 'driver_podium_5',
        'team_points_5', 'circuit_finish_3', 'starts']

driver_hist = defaultdict(lambda: deque(maxlen=5))
team_hist = defaultdict(lambda: deque(maxlen=5))
circuit_hist = defaultdict(lambda: deque(maxlen=3))


def avg(records, key):
    return float(np.mean([r[key] for r in records])) if records else np.nan


def form(driver, team, circuit):
    return dict(zip(FORM, [avg(driver_hist[driver], 'finish'), avg(driver_hist[driver], 'points'),
                           avg(driver_hist[driver], 'podium'), avg(team_hist[team], 'points'),
                           avg(circuit_hist[(driver, circuit)], 'finish'), len(driver_hist[driver])]))


rows = []
for (season, rnd), race in races.groupby(['season', 'round'], sort=True):
    for r in race.itertuples():
        rows.append({**form(r.driverId, r.constructorId, r.circuitId),
                     'grid': r.grid, 'practice_gap': r.practice_gap, 'quali_rain': r.quali_rain,
                     'race': f'{season}-{rnd:02d}', 'driver': r.driverCode,
                     'finish': r.position, 'podium': int(r.position <= 3)})
    # update the histories after the race, so a race never sees its own result
    for r in race.itertuples():
        record = dict(finish=float(r.position), points=float(r.points), podium=int(r.position <= 3))
        driver_hist[r.driverId].append(record)
        circuit_hist[(r.driverId, r.circuitId)].append(record)
    for team, team_race in race.groupby('constructorId'):
        team_hist[team].append(dict(points=float(team_race['points'].sum())))

data = pd.DataFrame(rows)
data = data[data['race'] >= '2023']
print(len(data), 'driver results from', data['race'].nunique(), 'races')

1794 driver results from 88 races


## Testing the models

I walk through every race from 2025 on. For each one the model is trained on the earlier races only, then asked for the podium of that race. The score is how many of its 3 picks really finished on the podium, averaged over the races (3 would be perfect).

The comparison to beat is simple: "the top 3 on the grid will be the podium". Each model is tried on five sets of inputs to see which inputs actually help.

In [7]:
def with_imputer(model):
    return make_pipeline(SimpleImputer(strategy='median'), model)


MODELS = {
    'Logistic Regression': lambda: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                                 LogisticRegression(max_iter=2000)),
    'Decision Tree': lambda: with_imputer(DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42)),
    'Random Forest': lambda: with_imputer(RandomForestClassifier(n_estimators=200, max_depth=4,
                                                                 min_samples_leaf=5, random_state=42)),
    'Extra Trees': lambda: with_imputer(ExtraTreesClassifier(n_estimators=200, max_depth=4,
                                                             min_samples_leaf=5, random_state=42)),
    'Gradient Boosting': lambda: with_imputer(GradientBoostingClassifier(n_estimators=100, max_depth=2,
                                                                         random_state=42)),
    'HistGradientBoosting': lambda: HistGradientBoostingClassifier(max_depth=3, max_iter=100,
                                                                    learning_rate=0.05, random_state=42),
    'XGBoost': lambda: XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=42),
}

INPUTS = {
    'grid only': ['grid'],
    'form only (no grid)': FORM,
    'grid + form': ['grid'] + FORM,
    'grid + form + practice': ['grid'] + FORM + ['practice_gap'],
    'grid + form + practice + rain': ['grid'] + FORM + ['practice_gap', 'quali_rain'],
}

test_races = [r for r in sorted(data['race'].unique()) if r >= '2025']
hits = defaultdict(list)
for race_id in test_races:
    train, test = data[data['race'] < race_id], data[data['race'] == race_id]
    real = set(test.nsmallest(3, 'finish')['driver'])
    hits['Top 3 on the grid'].append(len(real & set(test.nsmallest(3, 'grid')['driver'])))
    for input_name, cols in INPUTS.items():
        for model_name, make in MODELS.items():
            chance = make().fit(train[cols], train['podium']).predict_proba(test[cols])[:, 1]
            picks = test.assign(chance=chance).nlargest(3, 'chance')['driver']
            hits[f'{model_name} | {input_name}'].append(len(real & set(picks)))

results = pd.DataFrame({name: {'avg right of 3': np.mean(h), 'all 3 right': sum(x == 3 for x in h)}
                        for name, h in hits.items()}).T.round(2)
results = results.sort_values(['avg right of 3', 'all 3 right'], ascending=False)
print(len(test_races), 'races tested')
results.head(10)

42 races tested


,avg right of 3,all 3 right
HistGradientBoosting | grid + form,2.19,14.0
Decision Tree | grid + form,2.19,13.0
Decision Tree | grid + form + practice,2.19,13.0
Decision Tree | grid + form + practice + rain,2.19,13.0
XGBoost | grid + form + practice + rain,2.19,13.0
Gradient Boosting | grid + form,2.17,14.0
XGBoost | grid + form,2.17,14.0
Gradient Boosting | grid + form + practice,2.17,13.0
Random Forest | grid + form,2.14,12.0
XGBoost | grid + form + practice,2.14,12.0


In [8]:
# average score for each set of inputs, over all 7 models
by_inputs = {name: np.mean([np.mean(hits[f'{m} | {name}']) for m in MODELS]) for name in INPUTS}
print(pd.Series(by_inputs).round(2).to_string())
print('top 3 on the grid:', round(np.mean(hits['Top 3 on the grid']), 2))

grid only                        2.10
form only (no grid)              1.73
grid + form                      2.13
grid + form + practice           2.10
grid + form + practice + rain    2.13
top 3 on the grid: 2.1


## Picking the model

The best score wins, and a tie goes to the model that got all 3 right more often. The gap between the top models is small over this many races, so I also check the percentages the winner gives. Log loss scores them (lower is better), and the baseline is just the historical podium rate of each grid slot.

In [9]:
candidates = results.drop(index='Top 3 on the grid')
best = candidates.index[0]
model_name, input_name = best.split(' | ')
cols = INPUTS[input_name]
print('best:', best, '->', candidates.iloc[0]['avg right of 3'], 'of 3',
      '(grid rule:', results.loc['Top 3 on the grid', 'avg right of 3'], ')')

backtest = []
for race_id in test_races:
    train, test = data[data['race'] < race_id], data[data['race'] == race_id]
    fitted = MODELS[model_name]().fit(train[cols], train['podium'])
    backtest.append(test.assign(chance=fitted.predict_proba(test[cols])[:, 1]))
backtest = pd.concat(backtest)

slot_rate = data[data['race'] < '2025'].groupby('grid')['podium'].mean()
print('log loss, model:', round(log_loss(backtest['podium'], backtest['chance'].clip(0.01, 0.99)), 4))
print('log loss, grid slot history:',
      round(log_loss(backtest['podium'], backtest['grid'].map(slot_rate).fillna(0.01).clip(0.01, 0.99)), 4))

deep = backtest[backtest['grid'] >= 12]
print(f"started 12th or worse: {len(deep)} cases, {int(deep['podium'].sum())} podiums")
strong = data[(data['grid'] >= 12) & (data['driver_finish_5'] <= 5)]
print(f"strong recent form but started 12th or worse: {len(strong)} cases, {int(strong['podium'].sum())} podiums")

best: HistGradientBoosting | grid + form -> 2.19 of 3 (grid rule: 2.1 )


log loss, model: 0.198
log loss, grid slot history: 0.2052
started 12th or worse: 413 cases, 4 podiums
strong recent form but started 12th or worse: 23 cases, 5 podiums


## This weekend

The grid comes from the OpenF1 qualifying result, with penalties from the settings cell applied. Practice gaps and the qualifying rain share come from the same weekend.

In [10]:
quali = openf1('session_result', session_key=QUALI_KEY)
names = openf1('drivers', session_key=QUALI_KEY)[['driver_number', 'name_acronym', 'team_name']]
now = quali.merge(names, on='driver_number').rename(columns={'position': 'grid', 'name_acronym': 'driver'})
now['grid'] = [int(PENALTIES.get(d, g)) for d, g in zip(now['driver'], now['grid'])]

gaps = practice_gaps(PRACTICE_KEYS)
now['practice_gap'] = now['driver_number'].map(gaps)
now['quali_rain'] = rain_share(QUALI_KEY)

# link OpenF1 drivers to the ids used by the results (rookies with no history get empty form)
ids = races.drop_duplicates('driverCode', keep='last').set_index('driverCode')[['driverId', 'constructorId']]
now = now.join(ids, on='driver')
now['driverId'] = now['driverId'].fillna(now['driver'])
now['constructorId'] = now['constructorId'].fillna(now['team_name'])
now = pd.concat([now.reset_index(drop=True),
                 pd.DataFrame([form(r.driverId, r.constructorId, CIRCUIT) for r in now.itertuples()])], axis=1)
print(f"rain share in qualifying: {now['quali_rain'].iloc[0]:.0%}")

rain share in qualifying: 0%


In [11]:
model = MODELS[model_name]().fit(data[cols], data['podium'])
now['podium_chance'] = model.predict_proba(now[cols])[:, 1]
prediction = now.sort_values('podium_chance', ascending=False).reset_index(drop=True)
prediction.index += 1

print('model used:', best)
prediction[['driver', 'team_name', 'grid', 'podium_chance']].head(8).round(2)

model used: HistGradientBoosting | grid + form


,driver,team_name,grid,podium_chance
1,RUS,Mercedes,2,0.73
2,VER,Red Bull Racing,5,0.69
3,HAM,Ferrari,1,0.69
4,ANT,Mercedes,4,0.59
5,LEC,Ferrari,3,0.43
6,HAD,Red Bull Racing,7,0.13
7,NOR,McLaren,6,0.09
8,PIA,McLaren,12,0.02


In [12]:
Path('../predictions').mkdir(exist_ok=True)
prediction[['driver', 'team_name', 'grid', 'podium_chance']].round(3).to_csv(f'../predictions/{OUTPUT_NAME}.csv')

## Notes

- The starting grid carries most of the signal, so the picks stay fairly close to it.
- The differences between the best models are small. Treat the result as a ranking, not a promise.
- Penalties are typed in by hand from news and race control, so check them.
- The weather forecast for race day is not a model input.